# Lab 01 — Tokenization

**Goal:** understand what a token is and how different tokenizers split the same sentence differently.

In [1]:
# Our test sentence — we'll use this throughout the lab
text = "The cat sat on the mat. It wasn't moving."

# Approach 1: split on whitespace
tokens = text.split()
print(tokens)

['The', 'cat', 'sat', 'on', 'the', 'mat.', 'It', "wasn't", 'moving.']


In [2]:
# Approach 2: regex (regular expression) — word characters OR single non-word non-space characters
import re
tokens = re.findall(r"\w+|[^\w\s]", text)
print(tokens)

['The', 'cat', 'sat', 'on', 'the', 'mat', '.', 'It', 'wasn', "'", 't', 'moving', '.']


---

## How to have a productive dialogue with Qwen

Before we continue, learn to *ask well*. The quality of what you learn depends on the quality of your questions. Try these patterns:

### 1. Ask *why*, not *what*

- ❌ *"What is regex?"* → generic textbook answer, you already knew this
- ✅ *"Why did `wasn't` become three tokens with regex but stayed as one token with whitespace?"* → forces the model to reason about our specific code

### 2. Show the model your code and output

- ❌ *"Is my code correct?"* (no context — model guesses)
- ✅ *"I ran `re.findall(r'\w+|[^\w\s]', text)` and got `['wasn', "'", 't']`. Why did the apostrophe get its own token?"*

### 3. Ask for comparisons

- *"Compare whitespace and regex tokenization for the sentence `The U.S.A. is 250 years old.` — which handles it better?"*
- *"What would each of these tokenizers do with an email address `user@example.com`?"*

### 4. Push back when you disagree

- *"You said regex is better than whitespace, but it split `wasn't` into three pieces. Isn't that worse?"*
- The model will refine its answer. This is where real understanding happens.

### 5. Ask for edge cases

- *"Give me a sentence where regex tokenization would fail badly."*
- *"What is a case where whitespace tokenization is actually better than regex?"*

### 6. Verify by running

- The model can be wrong. If it says *"regex splits `don't` into `don` and `'t`"*, **try it in a new cell** and check.
- Running the model's suggestions is how you learn *and* catch its mistakes.

### One question to try right now

Copy this into your chat and see what Qwen says:

> *I have two outputs. Whitespace tokenizer: `['wasn't']` (1 token). Regex tokenizer: `['wasn', "'", 't']` (3 tokens). NLTK will give me `['was', "n't"]` (2 tokens). Rank these three from worst to best for a downstream sentiment classifier, and explain your reasoning.*

That's a real research question. There isn't one right answer. See how the model handles it.

from nlpa_chat import chat
_ = chat()

In [3]:
# Approach 3: NLTK — linguistically aware, Penn Treebank rules
!conad install nltk
#!pip install nltk
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize
print(word_tokenize(text))

['The', 'cat', 'sat', 'on', 'the', 'mat', '.', 'It', 'was', "n't", 'moving', '.']


In [4]:
# Compare all three tokenizers side by side
ws = text.split()
rgx = re.findall(r"\w+|[^\w\s]", text)
nlt = word_tokenize(text)
for i in range(max(len(ws), len(rgx), len(nlt))):
    a = ws[i] if i < len(ws) else ''
    b = rgx[i] if i < len(rgx) else ''
    c = nlt[i] if i < len(nlt) else ''
    print(f"{i:>2} | {a:<10} | {b:<10} | {c:<10}")

 0 | The        | The        | The       
 1 | cat        | cat        | cat       
 2 | sat        | sat        | sat       
 3 | on         | on         | on        
 4 | the        | the        | the       
 5 | mat.       | mat        | mat       
 6 | It         | .          | .         
 7 | wasn't     | It         | It        
 8 | moving.    | wasn       | was       
 9 |            | '          | n't       
10 |            | t          | moving    
11 |            | moving     | .         
12 |            | .          |           


chat()

In [5]:
# How many tokens did each approach produce?
print(f"Whitespace: {len(ws):>3}")
print(f"Regex:      {len(rgx):>3}")
print(f"NLTK:       {len(nlt):>3}")

Whitespace:   9
Regex:       13
NLTK:        12


In [1]:
from nlpa_chat import chat
chat()

In [7]:
# Approach 4: spaCy — industrial NLP standard
import spacy
nlp = spacy.load("en_core_web_sm")
spc = [t.text for t in nlp(text)]
print(spc)

['The', 'cat', 'sat', 'on', 'the', 'mat', '.', 'It', 'was', "n't", 'moving', '.']


In [8]:
chat()

# Moses tokenizer

### Moses was the tokenizer of statistical machine translation from the 2000s-2010s. Google Translate, Systran, all the early neural MT systems started here. sacremoses is the Python port.

In [10]:
# Approach 5: Moses — statistical MT workhorse (2000s-2010s)
from sacremoses import MosesTokenizer
mt = MosesTokenizer(lang='en')
mos = mt.tokenize(text)
print(mos)

['The', 'cat', 'sat', 'on', 'the', 'mat', '.', 'It', 'wasn', '&apos;t', 'moving', '.']


In [11]:
chat()

In [ ]:
#@cell N (Moses) @cell M (spaCy) — which output would be easier to use in a modern deep learning model, and why?
#@last if I wanted to prevent the HTML escaping, how would I do it?

# Stanza tokenizer

### Stanza is Stanford NLP's Python-native successor to CoreNLP. Unlike NLTK (hand-written rules) and spaCy (mixed rules + statistics), Stanza uses a neural network trained on Universal Dependencies treebanks to segment sentences and words. It's the "academic gold standard" for multilingual tokenization — supports 70+ languages out of the box with the same architecture.

In [12]:
# Approach 6: Stanza — Stanford's neural tokenizer, 70+ languages
import stanza
snlp = stanza.Pipeline(lang='en', processors='tokenize', verbose=False)
stz = [w.text for sent in snlp(text).sentences for w in sent.words]
print(stz)

['The', 'cat', 'sat', 'on', 'the', 'mat', '.', 'It', 'was', "n't", 'moving', '.']


# A summary: all 6 tokenizers, 5 sentences

In [13]:
# Session A summary: compare all 6 classical tokenizers on 5 tricky sentences
def tokenize_all_six(s):
    return {
        'ws':  s.split(),
        'rgx': re.findall(r"\w+|[^\w\s]", s),
        'nlt': word_tokenize(s),
        'spc': [t.text for t in nlp(s)],
        'mos': mt.tokenize(s),
        'stz': [w.text for sent in snlp(s).sentences for w in sent.words],
    }

# The comparison table

In [16]:
# Token counts per sentence across all 6 tokenizers
# Five deliberately tricky sentences that expose tokenizer differences
sentences = [
    "Dr. Smith paid $3.14 to buy 2 kg of apples at Whole Foods.",
    "I'll email you at john.doe@example.com by 5pm—don't be late!",
    "The COVID-19 pandemic (2019-2023) affected 7.5 billion people worldwide.",
    "She said, \"It's a state-of-the-art AI model,\" and smiled 😊.",
    "Visit https://example.org/page?id=42&lang=en for more info.",
]
print(f"{'Sentence':<10} {'WS':>4} {'RGX':>4} {'NLTK':>5} {'spaCy':>6} {'Moses':>6} {'Stanza':>7}")
print("-" * 48)
for i, s in enumerate(sentences, 1):
    tk = tokenize_all_six(s)
    print(f"S{i:<9} {len(tk['ws']):>4} {len(tk['rgx']):>4} "
          f"{len(tk['nlt']):>5} {len(tk['spc']):>6} "
          f"{len(tk['mos']):>6} {len(tk['stz']):>7}")

Sentence     WS  RGX  NLTK  spaCy  Moses  Stanza
------------------------------------------------
S1           13   18    15     15     15      15
S2            9   22    16     13     16      15
S3            9   18    12     14     12      16
S4           10   23    16     22     16      21
S5            5   22    12      6     20       6


# Pick a sentence and see all six tokenizers

In [17]:
# Look at ONE sentence across all 6 tokenizers
# Change idx to see different sentences (0 to 4)
idx = 4
s = sentences[idx]
tk = tokenize_all_six(s)
print(f"Sentence {idx+1}: {s}\n")
for name, tokens in tk.items():
    print(f"{name:>4} ({len(tokens):>2}): {tokens}")

Sentence 5: Visit https://example.org/page?id=42&lang=en for more info.

  ws ( 5): ['Visit', 'https://example.org/page?id=42&lang=en', 'for', 'more', 'info.']
 rgx (22): ['Visit', 'https', ':', '/', '/', 'example', '.', 'org', '/', 'page', '?', 'id', '=', '42', '&', 'lang', '=', 'en', 'for', 'more', 'info', '.']
 nlt (12): ['Visit', 'https', ':', '//example.org/page', '?', 'id=42', '&', 'lang=en', 'for', 'more', 'info', '.']
 spc ( 6): ['Visit', 'https://example.org/page?id=42&lang=en', 'for', 'more', 'info', '.']
 mos (20): ['Visit', 'https', ':', '/', '/', 'example.org', '/', 'page', '?', 'id', '=', '42', '&amp;', 'lang', '=', 'en', 'for', 'more', 'info', '.']
 stz ( 6): ['Visit', 'https://example.org/page?id=42&lang=en', 'for', 'more', 'info', '.']


# Side-by-side aligned view for a chosen sentence

In [18]:
# Position-by-position comparison for the same sentence
max_len = max(len(v) for v in tk.values())
print(f"Sentence {idx+1}: {s}\n")
print(f"{'#':>3}  {'WS':<40}  {'RGX':<15}  {'NLTK':<20}  {'spaCy':<40}  {'Moses':<15}  {'Stanza':<40}")
print("-" * 180)
for i in range(max_len):
    row = [tk[k][i] if i < len(tk[k]) else '' for k in ['ws','rgx','nlt','spc','mos','stz']]
    print(f"{i:>3}  {row[0]:<40}  {row[1]:<15}  {row[2]:<20}  {row[3]:<40}  {row[4]:<15}  {row[5]:<40}")

Sentence 5: Visit https://example.org/page?id=42&lang=en for more info.

  #  WS                                        RGX              NLTK                  spaCy                                     Moses            Stanza                                  
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0  Visit                                     Visit            Visit                 Visit                                     Visit            Visit                                   
  1  https://example.org/page?id=42&lang=en    https            https                 https://example.org/page?id=42&lang=en    https            https://example.org/page?id=42&lang=en  
  2  for                                       :                :                     for                                       :                for                                     
  

---

# Session B — Learned tokenizers

The six tokenizers we just built (whitespace, regex, NLTK, spaCy, Moses, Stanza) are all **rule-based**:

- Someone wrote rules for what a "token" is
- Those rules were carefully crafted for **one language at a time**
- Adding a new language (say, Telugu or Swahili) requires new rules

Around 2015-2016, NLP researchers proposed a completely different idea:

> **What if we let the tokenizer learn its own vocabulary from data?**

Instead of humans writing rules, an algorithm looks at millions of sentences and figures out which character sequences to keep together. Common words become single tokens. Rare words split into pieces. The vocabulary is learned, not designed.

This is **subword tokenization**. Every modern large language model — GPT, Claude, Llama, Qwen, Gemini, Mistral — uses some variant of it.

The main algorithms we will explore:

| Algorithm | Year | Used by |
|---|---|---|
| **BPE** (Byte-Pair Encoding) | 2015 | Original GPT, RoBERTa |
| **WordPiece** | 2016 | BERT, DistilBERT |
| **SentencePiece Unigram** | 2018 | T5, XLM-R, Llama |
| **Byte-level BPE** | 2019 | GPT-2, GPT-3 |
| **Tiktoken** | 2022 | GPT-4, GPT-4o |
| **Qwen's tokenizer** | 2024 | Qwen (running on this machine!) |

The final one is special: **it's the tokenizer inside the AI we've been talking to all lab**. When you type a message to Qwen in the chat widget, it uses this tokenizer to convert your text into numbers before doing anything else.

# BPE (Byte-Pair Encoding), the original

## BPE started as a compression algorithm in 1994. Sennrich et al. rediscovered it for NLP in 2015 and it changed everything. The algorithm is beautifully simple:

# Start with all characters as tokens
# Find the most frequent pair of adjacent tokens
# Merge them into a new token
# Repeat thousands of times

In [19]:
# Approach 7: BPE — the original learned tokenizer (Sennrich 2015, used by GPT-2)
from transformers import AutoTokenizer
bpe = AutoTokenizer.from_pretrained("gpt2")
tokens = bpe.tokenize(text)
print(tokens)

['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', '.', 'ĠIt', 'Ġwasn', "'t", 'Ġmoving', '.']


In [20]:
chat()

# WordPiece (BERT's tokenizer)
### WordPiece was introduced by Google in 2016 for their machine translation system, then made famous by BERT in 2018. The algorithm is very similar to BPE — start with characters, merge frequent pairs — but with one key difference: BPE merges by frequency, WordPiece merges by likelihood (which pair, if merged, would most improve a language model's probability of the training data).

In [21]:
# Approach 8: WordPiece — Google's variant, used by BERT (2018)
wp = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = wp.tokenize(text)
print(tokens)

['the', 'cat', 'sat', 'on', 'the', 'mat', '.', 'it', 'wasn', "'", 't', 'moving', '.']


In [22]:
print(wp.tokenize("tokenization"))

['token', '##ization']


In [23]:
#@cell N (WordPiece) — why does WordPiece lowercase everything but GPT-2's BPE doesn't?
#@cell N (WordPiece) @cell M (BPE) — both are learned tokenizers, both split into similar counts, but the outputs look completely different. Which format is easier for a downstream model to work with?
#@last what happens if I tokenize a made-up word like "kudocranium" with BERT?

# SentencePiece Unigram (T5's tokenizer)

### SentencePiece was developed by Google in 2018 (Taku Kudo). It's not really a single algorithm — it's a framework that can implement either BPE or a different algorithm called Unigram Language Model tokenization. T5, XLM-R, and Llama use the Unigram variant.

In [24]:
# Approach 9: SentencePiece Unigram — Kudo 2018, used by T5, XLM-R, Llama
sp = AutoTokenizer.from_pretrained("t5-small")
tokens = sp.tokenize(text)
print(tokens)

['▁The', '▁cat', '▁', 's', 'at', '▁on', '▁the', '▁mat', '.', '▁It', '▁wasn', "'", 't', '▁moving', '.']


In [25]:
# SentencePiece works identically on any language, including Telugu The "language-agnostic" claim in action
telugu = "నేను భారతదేశంలో నివసిస్తున్నాను."
print("English:", sp.tokenize(text))
print("Telugu :", sp.tokenize(telugu))

English: ['▁The', '▁cat', '▁', 's', 'at', '▁on', '▁the', '▁mat', '.', '▁It', '▁wasn', "'", 't', '▁moving', '.']
Telugu : ['▁', 'నేను', '▁', 'భారతదేశంలో', '▁', 'నివసిస్తున్నాను', '.']


In [26]:
#@cell N (SentencePiece) — what's the difference between ▁ (SentencePiece) and Ġ (BPE) and ## (WordPiece)? Do they all mean the same thing?
#@cell N (Telugu) — how many Telugu tokens vs English tokens for similar-length sentences? Why the difference?
#@last if SentencePiece is language-agnostic, why doesn't every model just use it?

# Byte-level BPE (GPT-2's variant)

Standard BPE has a subtle but fatal problem: what if the input contains a character the tokenizer has never seen before? Say a rare Unicode emoji, a Korean hangul the training data missed, or a symbol from ancient Aramaic. Traditional BPE would return an "unknown token" (UNK) — losing information forever.

Byte-level BPE, invented by OpenAI for GPT-2 in 2019, solves this with a beautifully radical idea:

What if instead of tokenizing characters, we tokenize the underlying bytes?

Every possible Unicode text is stored as a sequence of bytes (0-255). There are only 256 possible byte values, all known in advance. If BPE operates on bytes rather than characters, there is no such thing as an unknown input. Any Unicode text — English, Chinese, emoji, hieroglyphs, made-up symbols — can be tokenized without loss.

The "byte-level" tokenizer you already loaded is actually GPT-2's bpe from Step 15. It IS byte-level. Let me show you the difference dramatically.

In [27]:
# Approach 10: Byte-level BPE — GPT-2's trick, handles any Unicode
weird = "Hello 你好 مرحبا 🚀 Ω"
tokens = bpe.tokenize(weird)
print(f"Input:  {weird}")
print(f"Tokens: {tokens}")
print(f"Count:  {len(tokens)}")

Input:  Hello 你好 مرحبا 🚀 Ω
Tokens: ['Hello', 'Ġ', 'ä½', 'ł', 'å¥', '½', 'ĠÙħ', 'Ø±', 'Ø', 'Ń', 'Ø¨', 'Ø§', 'ĠðŁ', 'ļ', 'Ģ', 'ĠÎ', '©']
Count:  17


#  Read that carefully — the tokens look like garbage. That's the price of byte-level BPE. The Chinese 你好 shows up as Ġä½ ł å¥½ — those aren't wrong tokens, they're UTF-8 bytes displayed as if they were Latin characters. Chinese 你 is 3 bytes (E4 BD A0), each of which GPT-2 renders as a visible Latin symbol.

In [28]:
print(bpe.decode(bpe.encode(weird))) #You'll get back the original

Hello 你好 مرحبا 🚀 Ω


# Tiktoken (OpenAI's production tokenizer for GPT-4)

## Tiktoken is the tokenizer OpenAI ships with GPT-4 and GPT-4o. It's the same byte-level BPE algorithm as GPT-2 — but rewritten in Rust for speed. Where the HuggingFace transformers tokenizer for GPT-2 might tokenize a million tokens in 20 seconds, tiktoken does it in under 1 second. This matters because OpenAI runs it billions of times per day.

## The interesting thing is not the speed — it's that they can directly count tokens the way OpenAI's billing does. When you see "GPT-4 charges $30 per 1M input tokens", the definition of "token" is exactly what tiktoken counts.

In [29]:
# Approach 11: Tiktoken — OpenAI's production tokenizer for GPT-4 (fast, Rust-based)
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(text)
token_pieces = [enc.decode([tid]) for tid in token_ids]
print(f"Token IDs   : {token_ids}")
print(f"Token pieces: {token_pieces}")
print(f"Count       : {len(token_ids)}")

Token IDs   : [976, 9059, 10139, 402, 290, 2450, 13, 1225, 18101, 10067, 13]
Token pieces: ['The', ' cat', ' sat', ' on', ' the', ' mat', '.', ' It', " wasn't", ' moving', '.']
Count       : 11


# The practical token-cost demonstration

In [30]:
# Real-world use: estimating GPT-4 API costs before making a call
long_text = """
Natural language processing is a subfield of linguistics, computer science, 
and artificial intelligence concerned with the interactions between computers 
and human language. In particular, how to program computers to process and 
analyze large amounts of natural language data.
"""

tokens = enc.encode(long_text)
n_tokens = len(tokens)
cost_per_1M = 2.50  # GPT-4o input cost as of 2026, in USD

print(f"Text length : {len(long_text)} characters")
print(f"Token count : {n_tokens}")
print(f"Ratio       : {len(long_text)/n_tokens:.2f} chars per token")
print(f"Cost (input): ${n_tokens / 1_000_000 * cost_per_1M:.6f} USD")

Text length : 281 characters
Token count : 50
Ratio       : 5.62 chars per token
Cost (input): $0.000125 USD


# Qwen's actual tokenizer (the grand finale)

## This is the payoff of the entire lab. Every question students have asked Qwen through the chat widget went through this tokenizer first. Every response streamed back came out of it in reverse. It's not an abstraction — it's the code running on their machine right now.

## Qwen uses a byte-level BPE tokenizer (same family as GPT-2 and tiktoken) with a vocabulary of 151,643 tokens — the largest of any tokenizer we've seen. Why so large? Because Qwen was trained on multilingual data plus lots of code, and wanted efficient token counts for all of it.

In [31]:
# Approach 12: Qwen's actual tokenizer — the one inside our local AI
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
token_ids = qwen.encode(text)
token_pieces = [qwen.decode([tid]) for tid in token_ids]
print(f"Token count     : {len(token_ids)}")
print(f"Vocabulary size : {qwen.vocab_size:,} tokens")
print(f"Token IDs       : {token_ids}")
print(f"Token pieces    : {token_pieces}")

Token count     : 12
Vocabulary size : 151,643 tokens
Token IDs       : [785, 8251, 7578, 389, 279, 5517, 13, 1084, 5710, 944, 7218, 13]
Token pieces    : ['The', ' cat', ' sat', ' on', ' the', ' mat', '.', ' It', ' wasn', "'t", ' moving', '.']


# The full family comparison — one sentence, all six learned tokenizers

In [32]:
# All learned tokenizers on the same sentence
learned = {
    'GPT-2 (BPE)':          bpe.tokenize(text),
    'BERT (WordPiece)':     wp.tokenize(text),
    'T5 (SentencePiece)':   sp.tokenize(text),
    'GPT-4 (tiktoken)':     [enc.decode([t]) for t in enc.encode(text)],
    'Qwen (byte-BPE)':      [qwen.decode([t]) for t in qwen.encode(text)],
}
for name, tokens in learned.items():
    print(f"{name:<22} ({len(tokens):>2}): {tokens}")

GPT-2 (BPE)            (12): ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', '.', 'ĠIt', 'Ġwasn', "'t", 'Ġmoving', '.']
BERT (WordPiece)       (13): ['the', 'cat', 'sat', 'on', 'the', 'mat', '.', 'it', 'wasn', "'", 't', 'moving', '.']
T5 (SentencePiece)     (15): ['▁The', '▁cat', '▁', 's', 'at', '▁on', '▁the', '▁mat', '.', '▁It', '▁wasn', "'", 't', '▁moving', '.']
GPT-4 (tiktoken)       (11): ['The', ' cat', ' sat', ' on', ' the', ' mat', '.', ' It', " wasn't", ' moving', '.']
Qwen (byte-BPE)        (12): ['The', ' cat', ' sat', ' on', ' the', ' mat', '.', ' It', ' wasn', "'t", ' moving', '.']


# The token-cost check on the multilingual sample

In [34]:
# How does Qwen tokenize different scripts?
samples = {
    'English':  "The quick brown fox jumps over the lazy dog.",
    'Code':     "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    'Telugu':   "నేను భారతదేశంలో నివసిస్తున్నాను. నాకు తెలుగు బాగా ఇష్టం.",
}
print(f"{'Language':<12} {'Chars':>6} {'Tokens':>7} {'Chars/token':>13}")
print("-" * 40)
for name, s in samples.items():
    n = len(qwen.encode(s))
    print(f"{name:<12} {len(s):>6} {n:>7} {len(s)/n:>13.2f}")

Language      Chars  Tokens   Chars/token
----------------------------------------
English          44      10          4.40
Code             72      23          3.13
Telugu           56      83          0.67


### Telugu is ~6x more expensive than English on the same LLM. This is not a bug or an oversight — it's the direct consequence of training on English-heavy data. Every Telugu-language application built on top of Qwen (or GPT, Claude, Llama) pays this tax.

---

# Lab 01 complete — what you built

You have now implemented and compared **12 tokenizers** spanning six decades of NLP research:

**Classical era (rule-based):**

1. Whitespace — the naive baseline
2. Regex — mechanical splitting
3. NLTK — Penn Treebank hand-written rules
4. spaCy — industrial standard with URL/email awareness
5. Moses — statistical MT workhorse with XML-safe escaping
6. Stanza — neural, multilingual, academic gold standard

**Modern era (learned from data):**

7. BPE — Sennrich 2015, the algorithm that started it all
8. WordPiece — Google's variant, used by BERT
9. SentencePiece Unigram — language-agnostic, used by T5 and Llama
10. Byte-level BPE — GPT-2's trick, handles any Unicode losslessly
11. Tiktoken — OpenAI's production-fast implementation
12. Qwen's tokenizer — the one inside the AI running on your machine

## The three big ideas to remember

1. **Tokenization is not a solved problem.** Which tokenizer to use depends on your language, your task, your budget, and your history.
2. **Modern LLMs converge on learned subword tokenization** — because it works across all languages, handles unknown words gracefully, and lets vocabulary size be a design choice.
3. **Not all languages are equal in the eyes of a tokenizer** — English gets ~4 chars per token, Indian languages often get ~0.5-1. This has real cost, latency, and fairness implications.

## Where to go next

- **Lab 02** — embeddings: what happens after tokens become numbers?
- **Lab 03** — training your own tokenizer on a custom corpus
- **Lab 06 (research)** — building a fair tokenizer for Telugu

*This lab was built with the help of a local AI model running on the same machine you ran it on. No cloud, no cost, no data leaving your laptop.*

In [ ]:
chat()